# FER2013 — Facial Expression Recognition
**Iterative architecture experiments with WandB tracking**

Architectures: TinyCNN → SmallCNN → MediumCNN → ResNetStyleCNN → TransferLearning

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install wandb kaggle tqdm pyyaml scikit-learn seaborn -q

In [ ]:
# ── 2. Mount Drive & clone repo ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/fer-facial-expression-recognition'
if not os.path.exists(REPO):
    !git clone https://github.com/dachishonia/fer-facial-expression-recognition.git $REPO
else:
    !git -C $REPO pull

import sys
sys.path.insert(0, REPO)
sys.path.insert(0, f'{REPO}/src')
print('Repo ready.')

In [ ]:
# ── 3. Kaggle credentials & dataset download ─────────────────────────────────
import json, os

os.makedirs('/root/.kaggle', exist_ok=True)

# Fill in your credentials here
KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # <-- change this
KAGGLE_KEY      = 'YOUR_KAGGLE_API_KEY'   # <-- change this

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

DATA_DIR = '/content/fer2013'
CSV_PATH = f'{DATA_DIR}/fer2013.csv'

if not os.path.exists(CSV_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle competitions download \
        -c challenges-in-representation-learning-facial-expression-recognition-challenge \
        -p {DATA_DIR}
    !unzip -q {DATA_DIR}/*.zip -d {DATA_DIR}
    print('Dataset downloaded.')
else:
    print('Dataset already present.')

In [ ]:
# ── 4. WandB login ───────────────────────────────────────────────────────────
import wandb
wandb.login()

In [ ]:
# ── 5. GPU check & imports ───────────────────────────────────────────────────
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

from dataset import get_dataloaders, get_class_weights, EMOTION_LABELS
from models  import get_model, count_parameters
from train   import train_model, overfit_single_batch
from utils   import run_sanity_checks

In [ ]:
# ── 6. Explore the dataset ───────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f'Total samples : {len(df)}')
print(f'Splits        : {df["Usage"].value_counts().to_dict()}')
print()

train_df = df[df['Usage'] == 'Training']
counts = train_df['emotion'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(EMOTION_LABELS, counts.values, color='steelblue')
ax.set_title('Class distribution (Training set)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print('Note: Disgust is severely under-represented — class weights will compensate.')

In [ ]:
# ── 7. Visualise sample images ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for emotion_id in range(7):
    rows = train_df[train_df['emotion'] == emotion_id].sample(2)
    for row_idx, (_, row) in enumerate(rows.iterrows()):
        pixels = np.array(row['pixels'].split(), dtype=np.uint8).reshape(48, 48)
        axes[row_idx, emotion_id].imshow(pixels, cmap='gray')
        axes[row_idx, emotion_id].set_title(EMOTION_LABELS[emotion_id], fontsize=9)
        axes[row_idx, emotion_id].axis('off')
plt.suptitle('Sample images per class', y=1.02)
plt.tight_layout()
plt.show()

## Experiment 1 — TinyCNN (Intentional Underfitting Baseline)
2 conv layers, ~50K parameters. Expected to underfit — not enough capacity to learn 7 emotion classes.

In [ ]:
class_weights = get_class_weights(CSV_PATH)

# Run 1a: TinyCNN default
cfg = {
    'run_name': 'tiny_cnn_v1_adam_lr1e3', 'architecture': 'tiny_cnn',
    'epochs': 40, 'lr': 1e-3, 'optimizer': 'adam', 'scheduler': 'plateau',
    'batch_size': 64, 'dropout': 0.0, 'augment': False,
    'weight_decay': 0.0, 'for_transfer': False, 'early_stop_patience': 15,
}
train_loader, val_loader, test_loader = get_dataloaders(
    CSV_PATH, batch_size=cfg['batch_size'], augment=cfg['augment'])

model = get_model('tiny_cnn', dropout=cfg['dropout']).to(device)
total, trainable = count_parameters(model)
print(f'TinyCNN — total params: {total:,} | trainable: {trainable:,}')

print('\n[Sanity checks before training]')
run_sanity_checks(model, train_loader, device)

print('\n[Single-batch overfit test]')
overfit_single_batch(get_model('tiny_cnn').to(device), train_loader, device, steps=50)

model, best_acc = train_model(model, cfg, train_loader, val_loader, device, class_weights)
print(f'\nBest val acc: {best_acc:.3f}')

In [ ]:
# Run 1b: higher LR
cfg_1b = {**cfg, 'run_name': 'tiny_cnn_v2_adam_lr1e2', 'lr': 1e-2}
model_1b = get_model('tiny_cnn').to(device)
model_1b, acc_1b = train_model(model_1b, cfg_1b, train_loader, val_loader, device, class_weights)
print(f'Best val acc (lr=0.01): {acc_1b:.3f}')

In [ ]:
# Run 1c: SGD optimizer
cfg_1c = {**cfg, 'run_name': 'tiny_cnn_v3_sgd', 'lr': 1e-2,
          'optimizer': 'sgd', 'scheduler': 'step', 'weight_decay': 1e-4}
model_1c = get_model('tiny_cnn').to(device)
model_1c, acc_1c = train_model(model_1c, cfg_1c, train_loader, val_loader, device, class_weights)
print(f'Best val acc (SGD): {acc_1c:.3f}')

## Experiment 2 — SmallCNN (Overfitting Demonstration)
4 conv layers, ~1.2M params, **no regularization**. High train acc but large train/val gap = overfitting.

In [ ]:
# Run 2a: No regularization — will overfit
cfg2 = {
    'run_name': 'small_cnn_v1_no_reg', 'architecture': 'small_cnn',
    'epochs': 60, 'lr': 1e-3, 'optimizer': 'adam', 'scheduler': 'plateau',
    'batch_size': 64, 'dropout': 0.0, 'augment': False,
    'weight_decay': 0.0, 'for_transfer': False, 'early_stop_patience': 20,
}
train_loader2, val_loader2, _ = get_dataloaders(
    CSV_PATH, batch_size=cfg2['batch_size'], augment=cfg2['augment'])

model2 = get_model('small_cnn', dropout=0.0).to(device)
total, trainable = count_parameters(model2)
print(f'SmallCNN — total params: {total:,}')
run_sanity_checks(model2, train_loader2, device)

model2, acc2 = train_model(model2, cfg2, train_loader2, val_loader2, device, class_weights)
print(f'Best val acc (no reg): {acc2:.3f}')

In [ ]:
# Run 2b: Add dropout + augmentation to combat overfitting
cfg2b = {**cfg2, 'run_name': 'small_cnn_v2_dropout', 'dropout': 0.3,
         'augment': True, 'weight_decay': 1e-4}
train_loader2b, val_loader2b, _ = get_dataloaders(
    CSV_PATH, batch_size=cfg2b['batch_size'], augment=cfg2b['augment'])
model2b = get_model('small_cnn', dropout=0.3).to(device)
model2b, acc2b = train_model(model2b, cfg2b, train_loader2b, val_loader2b, device, class_weights)
print(f'Best val acc (dropout=0.3): {acc2b:.3f}')

## Experiment 3 — MediumCNN (BatchNorm + Dropout)
BatchNorm stabilises internal covariate shift; Dropout prevents co-adaptation. Testing dropout strength.

In [ ]:
for dropout_val, sched, bs, run_name in [
    (0.3, 'plateau', 64, 'medium_cnn_v1_dropout03'),
    (0.5, 'plateau', 64, 'medium_cnn_v2_dropout05'),
    (0.4, 'cosine',  32, 'medium_cnn_v3_cosine'),
]:
    cfg3 = {
        'run_name': run_name, 'architecture': 'medium_cnn',
        'epochs': 60, 'lr': 1e-3, 'optimizer': 'adam', 'scheduler': sched,
        'batch_size': bs, 'dropout': dropout_val, 'augment': True,
        'weight_decay': 1e-4, 'for_transfer': False, 'early_stop_patience': 15,
    }
    tl, vl, _ = get_dataloaders(CSV_PATH, batch_size=bs, augment=True)
    m = get_model('medium_cnn', dropout=dropout_val).to(device)
    total, _ = count_parameters(m)
    print(f'\nMediumCNN ({run_name}) — params: {total:,}')
    m, acc = train_model(m, cfg3, tl, vl, device, class_weights)
    print(f'Best val acc: {acc:.3f}')

## Experiment 4 — ResNetStyleCNN (Residual Connections)
Skip connections allow gradients to flow freely. Enables deeper network without vanishing gradient.

In [ ]:
for sched, run_name in [('plateau', 'resnet_style_v1_plateau'), ('cosine', 'resnet_style_v2_cosine')]:
    cfg4 = {
        'run_name': run_name, 'architecture': 'resnet_style',
        'epochs': 60, 'lr': 1e-3, 'optimizer': 'adam', 'scheduler': sched,
        'batch_size': 64, 'dropout': 0.4, 'augment': True,
        'weight_decay': 1e-4, 'for_transfer': False, 'early_stop_patience': 15,
    }
    tl, vl, _ = get_dataloaders(CSV_PATH, batch_size=64, augment=True)
    m = get_model('resnet_style', dropout=0.4).to(device)
    total, _ = count_parameters(m)
    print(f'\nResNetStyle ({run_name}) — params: {total:,}')
    run_sanity_checks(m, tl, device)
    m, acc = train_model(m, cfg4, tl, vl, device, class_weights)
    print(f'Best val acc: {acc:.3f}')

## Experiment 5 — Transfer Learning (MobileNetV2)
Pretrained ImageNet features. Grayscale images upsampled to 224×224 and converted to 3-channel.

In [ ]:
# Run 5a: Only last 3 blocks unfrozen
cfg5a = {
    'run_name': 'transfer_mobilenet_v1_frozen', 'architecture': 'transfer_learning',
    'epochs': 30, 'lr': 3e-4, 'optimizer': 'adam', 'scheduler': 'plateau',
    'batch_size': 32, 'dropout': 0.5, 'augment': True,
    'weight_decay': 1e-4, 'for_transfer': True, 'early_stop_patience': 10,
}
tl5, vl5, _ = get_dataloaders(CSV_PATH, batch_size=32, augment=True, for_transfer=True)
m5a = get_model('transfer_learning', dropout=0.5, freeze_until=-3).to(device)
total, trainable = count_parameters(m5a)
print(f'TransferCNN — total: {total:,} | trainable: {trainable:,}')
run_sanity_checks(m5a, tl5, device)
m5a, acc5a = train_model(m5a, cfg5a, tl5, vl5, device, class_weights)
print(f'Best val acc (frozen): {acc5a:.3f}')

In [ ]:
# Run 5b: Fine-tune more layers with lower LR
cfg5b = {
    'run_name': 'transfer_mobilenet_v2_finetune', 'architecture': 'transfer_learning',
    'epochs': 30, 'lr': 1e-4, 'optimizer': 'adam', 'scheduler': 'cosine',
    'batch_size': 32, 'dropout': 0.3, 'augment': True,
    'weight_decay': 5e-5, 'for_transfer': True, 'early_stop_patience': 10,
}
m5b = get_model('transfer_learning', dropout=0.3, freeze_until=-6).to(device)
total, trainable = count_parameters(m5b)
print(f'TransferCNN (fine-tune) — total: {total:,} | trainable: {trainable:,}')
m5b, acc5b = train_model(m5b, cfg5b, tl5, vl5, device, class_weights)
print(f'Best val acc (fine-tune): {acc5b:.3f}')

## Summary — All Experiments

In [ ]:
print('All experiments complete.')
print('View results at: https://wandb.ai/dshon23/fer-challenge')
print()
print('Expected progression:')
print('  TinyCNN      → ~35-40% val acc  (underfit: low capacity)')
print('  SmallCNN     → train>70%, val~45% (overfit: no regularization)')
print('  MediumCNN    → ~55-60% val acc  (better generalization)')
print('  ResNetStyle  → ~60-65% val acc  (residual connections help)')
print('  TransferNet  → ~65-70% val acc  (ImageNet features)')